# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ladinski/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

if token:
    print("HF_TOKEN loaded successfully!")
else:
    print("Token not found.")

HF_TOKEN loaded successfully!


## 1. Unit of Analysis + Time Window

**Selected lane:** Refresh / Content Opportunity Scoring

**Unit of analysis:** One web page during one monthly observation window, subject to verification of the warehouse table's actual grain.

**Data source:** FlyRank Internship Warehouse on Hugging Face, queried using DuckDB.

**Development window:** March 2026 (`month = 2026-03`).

**Reserved evaluation window:** June 2026, which will not be used for developing labels, selecting features, or tuning models.

**Research objective:** Identify and rank web pages that may deserve editorial review based on observable search-performance and content signals.

**Prediction target:** A binary indicator of declining search performance, provided a valid future outcome can be constructed from the warehouse data.

**Decision supported:** Which pages should an SEO specialist investigate first?

**Deliberate exclusion:** Future performance information and columns derived from the prediction target, because including them would introduce target leakage.

The exact table, date coverage, row uniqueness, and availability of features will be verified through warehouse queries before finalizing this data contract.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: Feature / Label / Context / Excluded

**Selected lane:** Refresh / Content Opportunity Scoring

The purpose of this data contract is to distinguish information available when making a recommendation from information used to evaluate that recommendation.

### Feature Fields — Model Inputs

I propose starting with five features:

1. **Search impressions:** Measures how frequently a page appears in search results.
2. **Average search position:** Describes a page's search visibility.
3. **Click-through rate (CTR):** Measures the proportion of impressions resulting in clicks.
4. **Content age:** Indicates how long the content has existed.
5. **Days since last update:** Measures content update recency.

These are candidate features. Their actual warehouse column names, availability, and observation windows must be verified before use.

### Label / Target

**Proposed target:** `is_declining_label`

This will be a binary indicator:

- `1` — A page experiences declining search performance during a subsequent outcome window.
- `0` — A page does not meet the defined decline criterion.

The label must be calculated using a clearly defined future observation period. Its construction will depend on the actual warehouse schema.

### Context Fields

Context fields will be used to understand, group, or validate observations rather than automatically becoming predictive features.

Examples include:

- Anonymized page identifier.
- Anonymized client identifier.
- Observation month or date.
- Data-source availability flags.

Client identifiers can support grouped validation to reduce leakage between clients.

### Excluded Fields

**Future performance measurements:** Excluded from input features because they would reveal information unavailable when making the recommendation.

**Label-derived fields:** Excluded because they could directly reveal the prediction target.

**Identifiers:** Page and client IDs will not automatically be used as predictive features because they may encourage memorization rather than generalizable patterns.

**Private information:** Client names, raw page URLs, and sensitive search queries will not be published.

### Feature Availability Rule

Every model feature must be observable at the recommendation time.

The final feature list will be confirmed against the warehouse schema and its availability flags.

A feature that is absent, unavailable, or derived from the future outcome will not be included in the honest model.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb huggingface_hub


import duckdb
from google.colab import userdata

# Read token securely from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# Warehouse tables
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print("DuckDB connected successfully!")



DuckDB connected successfully!


In [8]:

# Inspect the schema without loading the full dataset
schema = con.sql(
    f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}"
).df()

display(schema[["column_name", "column_type"]])


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [9]:

# Query 1: Verify that each row represents
# one client + one content page + one day.

query1 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (
        client_hash_id,
        content_hash_id,
        report_date
    )) AS unique_page_days,
    COUNT(*) - COUNT(DISTINCT (
        client_hash_id,
        content_hash_id,
        report_date
    )) AS duplicate_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""

result1 = con.sql(query1).df()
display(result1)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_page_days,duplicate_rows
0,9841378,9841378,0


In [10]:

# Query 2: Verify the number of pages and date range
# for the March 2026 development window.

query2 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (client_hash_id, content_hash_id))
        AS unique_pages,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT report_date) AS unique_days
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""

result2 = con.sql(query2).df()
display(result2)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_pages,first_date,last_date,unique_days
0,9841378,331437,2026-03-01,2026-03-31,31


In [11]:

# Query 3: Check GSC and GA4 data availability
# using the required IS TRUE condition.

query3 = f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS both_available_rows

FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""

result3 = con.sql(query3).df()

result3["gsc_available_pct"] = (
    result3["gsc_available_rows"] /
    result3["total_rows"] * 100
).round(2)

display(result3)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows,gsc_available_pct
0,9841378,3611061,413966,364347,36.69


In [12]:

# Build five page-level features from March 2026.
# Aggregate in DuckDB before converting to pandas.

feature_query = f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
),
monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_30d,
        SUM(gsc_clicks) AS clicks_30d,
        SUM(gsc_clicks) * 1.0 /
            NULLIF(SUM(gsc_impressions), 0) AS ctr_30d,
        SUM(gsc_avg_position * gsc_impressions) /
            NULLIF(
                SUM(
                    CASE WHEN gsc_avg_position IS NOT NULL
                         THEN gsc_impressions ELSE 0 END
                ), 0
            ) AS avg_position_30d,
        COUNT(DISTINCT report_date) AS active_days_30d
    FROM daily
    GROUP BY client_hash_id, content_hash_id
)
SELECT * FROM monthly
"""

features_df = con.sql(feature_query).df()

print("Pages in feature frame:", len(features_df))
display(features_df.head(10))

print("\nMissing values:")
print(features_df.isna().sum())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages in feature frame: 176738


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,ctr_30d,avg_position_30d,active_days_30d
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,0.001073,6.893301,31
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.214128,31
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,0.001066,6.535346,31
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,0.002629,7.435680,31
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,0.002331,3.871795,31
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,223.0,1.0,0.004484,10.538117,31
6,client_73cda7b4e4f265ea,content_1f380a642aed423b,96.0,1.0,0.010417,5.864583,31
7,client_73cda7b4e4f265ea,content_22c063002b7c1caf,314.0,1.0,0.003185,10.057325,31
8,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,7709.0,20.0,0.002594,5.127643,31
9,client_73cda7b4e4f265ea,content_20403327d8d9374c,3561.0,10.0,0.002808,9.273799,31



Missing values:
client_hash_id      0
content_hash_id     0
impressions_30d     0
clicks_30d          0
ctr_30d             0
avg_position_30d    0
active_days_30d     0
dtype: int64


### Five Features and Their Availability

**Decision moment:** April 1, 2026, assuming March data has finished syncing into the warehouse.

| Feature | Description | Why available at decision time |
|---|---|---|
| `impressions_30d` | Total GSC impressions in March | Historical March impressions are available after the reporting period closes. |
| `clicks_30d` | Total GSC clicks in March | Historical March clicks are observable before making the April recommendation. |
| `ctr_30d` | March clicks divided by impressions | Computed entirely from historical March measurements. |
| `avg_position_30d` | Impression-weighted average search position | Derived from March search-position and impression data. |
| `active_days_30d` | Number of March days with available GSC data | Computed from historical data-availability records. |

**Important:** March contains 31 days. The `_30d` suffix is provisional; these columns should be renamed `_mar2026` for accuracy.

All five features are intended to be knowable at the decision moment. In production, the actual decision time would need to account for warehouse synchronization delays.

In [13]:

# Check April data coverage without exposing client details.

april_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {TABLES['fact_daily']}
WHERE month = '2026-04'
""").df()

display(april_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,first_date,last_date
0,10424730,3901060,2026-04-01,2026-04-30


In [14]:

import numpy as np

# April outcomes: aggregate at page level
april_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS april_clicks,
    COUNT(DISTINCT report_date) AS april_gsc_days
FROM {TABLES['fact_daily']}
WHERE month = '2026-04'
  AND gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
"""

april_df = con.sql(april_query).df()

# Join April outcomes to March feature frame
model_df = features_df.merge(
    april_df,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    validate="one_to_one"
)

# Require observed GSC data on every day of both months
model_df = model_df[
    (model_df["active_days_30d"] == 31) &
    (model_df["april_gsc_days"] == 30) &
    (model_df["clicks_30d"] > 0)
].copy()

# Calculate the subsequent click change
model_df["future_click_change_pct"] = (
    (model_df["april_clicks"] - model_df["clicks_30d"])
    / model_df["clicks_30d"]
    * 100
)

# Binary target: April clicks declined by 20% or more
model_df["is_declining_label"] = (
    model_df["future_click_change_pct"] <= -20
).astype(int)

print("Eligible pages:", len(model_df))
print("\nTarget distribution:")
print(model_df["is_declining_label"].value_counts())
print("\nTarget percentages:")
print(
    model_df["is_declining_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(model_df[
    ["impressions_30d", "clicks_30d", "ctr_30d",
     "avg_position_30d", "active_days_30d",
     "is_declining_label"]
].head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible pages: 39938

Target distribution:
is_declining_label
1    23577
0    16361
Name: count, dtype: int64

Target percentages:
is_declining_label
1    59.03
0    40.97
Name: proportion, dtype: float64


,impressions_30d,clicks_30d,ctr_30d,avg_position_30d,active_days_30d,is_declining_label
0,6523.0,7.0,0.001073,6.893301,31,0
2,5630.0,6.0,0.001066,6.535346,31,1
3,4944.0,13.0,0.002629,7.435680,31,1
4,429.0,1.0,0.002331,3.871795,31,1
5,223.0,1.0,0.004484,10.538117,31,0


In [15]:

import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score

feature_cols = [
    "impressions_30d",
    "clicks_30d",
    "ctr_30d",
    "avg_position_30d",
    "active_days_30d"
]

# Keep clients separate between training and testing
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(splitter.split(
    model_df,
    model_df["is_declining_label"],
    groups=model_df["client_hash_id"]
))

y_train = model_df.iloc[train_idx]["is_declining_label"]
y_test = model_df.iloc[test_idx]["is_declining_label"]

def evaluate_tree(columns):
    X_train = model_df.iloc[train_idx][columns].fillna(0)
    X_test = model_df.iloc[test_idx][columns].fillna(0)

    model = DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    return {
        "Accuracy": round(accuracy_score(y_test, predictions), 3),
        "Precision": round(
            precision_score(y_test, predictions, zero_division=0), 3
        )
    }

# Model 1: Only features available in March
honest = evaluate_tree(feature_cols)

# Model 2: Deliberately include the future-derived answer
leaky = evaluate_tree(feature_cols + ["future_click_change_pct"])

print("Honest model:", honest)
print("Leaky model: ", leaky)

# Keep the honest feature list for subsequent modeling
print("\nFinal retained features:", feature_cols)


Honest model: {'Accuracy': 0.595, 'Precision': 0.666}
Leaky model:  {'Accuracy': 1.0, 'Precision': 1.0}

Final retained features: ['impressions_30d', 'clicks_30d', 'ctr_30d', 'avg_position_30d', 'active_days_30d']


### Deliberate Target Leakage Experiment

**Objective:** Demonstrate how including future outcome information can artificially inflate model performance.

I trained two depth-3 decision trees using a client-holdout split.

**Honest model:** Used five features calculated from March 2026 data.

**Leaky model:** Used the same features plus `future_click_change_pct`, which directly determines whether a page is labeled as declining in April.

| Metric | Honest model | Leaky model |
|---|---:|---:|
| Accuracy | 0.595 | 1.000 |
| Precision | 0.666 | 1.000 |

**Observation:** The leaky model achieved perfect performance because it had access to the future outcome used to construct the target.

Client-holdout validation alone cannot prevent this type of leakage.

**Correction:** I deliberately introduced the leaked feature to demonstrate the problem, then excluded it from the final model feature list.

The five retained features are March search impressions, clicks, CTR, average position, and number of GSC-available days.

**Conclusion:** A model must only use features available at the decision moment. High evaluation scores are not meaningful when future information leaks into the inputs.

In [16]:

# Rename March features for accurate documentation
rename_map = {
    "impressions_30d": "impressions_mar2026",
    "clicks_30d": "clicks_mar2026",
    "ctr_30d": "ctr_mar2026",
    "avg_position_30d": "avg_position_mar2026",
    "active_days_30d": "active_days_mar2026"
}

model_df = model_df.rename(columns=rename_map)

# Explicitly retain ONLY the five honest model features
final_features = list(rename_map.values())

X_final = model_df[final_features].copy()
y_final = model_df["is_declining_label"].copy()

assert X_final.shape[1] == 5
assert "future_click_change_pct" not in X_final.columns

print("Final feature count:", X_final.shape[1])
print("Future-derived feature excluded:",
      "future_click_change_pct" not in X_final.columns)

display(X_final.head())


Final feature count: 5
Future-derived feature excluded: True


,impressions_mar2026,clicks_mar2026,ctr_mar2026,avg_position_mar2026,active_days_mar2026
0,6523.0,7.0,0.001073,6.893301,31
2,5630.0,6.0,0.001066,6.535346,31
3,4944.0,13.0,0.002629,7.435680,31
4,429.0,1.0,0.002331,3.871795,31
5,223.0,1.0,0.004484,10.538117,31


In [17]:

print("Final features:", final_features)
print("Feature count:", len(final_features))
print("Leaked column excluded:",
      "future_click_change_pct" not in X_final.columns)

assert len(final_features) == 5
assert "future_click_change_pct" not in X_final.columns

print("Final checks passed!")


Final features: ['impressions_mar2026', 'clicks_mar2026', 'ctr_mar2026', 'avg_position_mar2026', 'active_days_mar2026']
Feature count: 5
Leaked column excluded: True
Final checks passed!


## 4. Data Limits

The analysis uses anonymized Google Search Console observations from March and April 2026.

The primary limitations are:

**1. Data availability and selection bias**

Only 3,611,061 of 9,841,378 March daily records (approximately 36.7%) have GSC data available.

The modeling cohort is further restricted to pages with complete GSC coverage in both months and positive March clicks, leaving 39,938 eligible pages. These pages may not represent all content in the warehouse.

**2. Unequal observation windows**

March has 31 days and April has 30. Comparing monthly click totals may overstate declines. A stronger version would use clicks per available day or equal-length windows.

**3. Proxy label limitations**

The target identifies pages with an observed decline of at least 20% in subsequent monthly clicks. It does not measure whether refreshing content would improve traffic.

**4. Missing content-level features**

The current feature frame uses GSC performance measurements rather than the full content metadata. Content age and update history may require joining additional warehouse tables.

**5. Generalization**

The honest model achieved 59.5% accuracy and 66.6% precision using a client-holdout split. These results are preliminary and should be compared against simple baselines using ranking metrics.

**6. Time-dependent validation**

Client-holdout validation helps evaluate generalization across clients, but a separate time-based evaluation is also necessary for forecasting future outcomes.

**Conclusion:** The warehouse can support a useful page-ranking experiment, but coverage limitations, label design, and temporal validity must be addressed before deployment.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.